## Encounter simulation: 1 player vs 1 enemy

### Using for environments:
* RL card games (https://github.com/datamllab/rlcard)
* Chess petting zoo (https://pettingzoo.farama.org/environments/classic/chess/)

---

## Strategy overview
1. Define the player and the enemy creature and pull out all base stats and full action list from data
2. Determine initative order (does enemy or player go first)

Start of encounter: ticker= 3 actions each 
3. Setup environment and define the current state

START OF EACH ROUND
* Distance of player and enemy to eachother -> determines if melee or range
* Current player stats -> HP, AC, other relevant stats, and calculate and current buffs or conditions that would impact those stats; current spell slots
* Current enemy stats -> same and player

4. Based on the state, filter what actions are possible
* Filter action list based on 1) distance to the enemy (melee/range), 2) Conditions that need to be met (i.e. off-gaurd for sneak attack), 3) action economy (spell slots, cost of each action)

5. Choose actions up to the 3 action cost (either RANDOMLY, MAX DAMAGE, or using Deep q-learning) and calculate 1) Success of hit, 2) total damage dealt, 3) impact on status of enemy (give a condition)
* For one round get both the player and the enemy action choices and results

6. Calculate the reward for that round: Goal is to optimize the action combinations that results in the LEAST number of rounds for an encounter to end
** Alternatively -> run the action choices until end of encounter (so either the enemy or player runs down to 0)
* Reward strategy 1= (0/1/-1) action success + total damage dealt + bonus if end of round (+ if player wins, -1 if monster wins)
* Reward strategy 2= Just the result of the whole encounter (+1 player wins, -1 monster wins) + total number of rounds for the encounter

7. Update the state based on actions (health, status) and repeat simulation util one of the two creatures health = 0 

As a result, the data I will have: 
1. How DIFFICULT is the monster for each player, based on the number of rounds it took to defeat and whether the moster defeated the player
2. What is the combination of attacks (for both enemy and player) that is MOST EFFECTIVE against the opponent

I would repeat the simulation for each player-monster combination for 20 rounds of combat (number arbitrary), then have a major dataset that has:
1. Difficulty ranking for each player-monster based on (data distribution) of number rounds and relative difficulty --> and the spread for how likely the minumum/optimal round encounter is for each combination
2. A list of effective attacks for the monster against specific players -> helpful for a DM running a campaign to know what attack actions to pick
3.(stretch goal/ fall back) Features of the attacks and monster details that could be used in a linear regression to predict if a certain factor of a monster makes it more or less difficult (i.e. a range fire monster is more challenging for a fighter)

The stretch goal:
* Use the difficulty of multple players in combination to see what the difficulty of a group of monsters would be
* Use a battle map layout

----

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gymnasium
#Import libraries 
import pandas as pd
import numpy as np
import requests
import matplotlib as plt
import seaborn as sns
import matplotlib.pyplot as plt


import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re
import random

import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=SettingWithCopyWarning)

### Import data for player

In [2]:
## import character build basic stats
build_skill = pd.read_csv('../build data/all_build_base_skills.csv')
char_skill = build_skill[build_skill['Unnamed: 0']== 'halfling rogue dexterity']
#rename columns to match monster format 
char_skill= char_skill.rename(columns={'hit points':'HP', 'armor class':'AC', 'Unnamed: 0':'character build'})

#import build stat modifiers
build_stats = pd.read_csv('../build data/all_build_base_stats.csv')
char_stats = build_stats[build_stats['Unnamed: 0']== 'halfling rogue dexterity']

char_full_build = pd.merge(char_stats, char_skill)

#condense build information for only the relevant information
short_col = ['character build', 'level', 'strength', 'dexterity',
       'constitution', 'intelligence', 'wisdom', 'charisma', 
       'HP', 'speed', 'perception', 'fortitude', 'reflex', 'will', 'AC', 'spell attack', 'spell dc']

char_full_build = char_full_build[short_col]

#convert information to a dictionary
char_skill_dict= char_full_build.iloc[0].to_dict()

## rename key terms to match rest of code 
char_skill_dict['hp']= char_skill_dict.pop('HP')
char_skill_dict['ac']= char_skill_dict.pop('AC')
char_skill_dict['speed']= char_skill_dict.pop('speed')
char_skill_dict['percep']= char_skill_dict.pop('perception')
char_skill_dict['fort']= char_skill_dict.pop('fortitude')
char_skill_dict['ref']= char_skill_dict.pop('reflex')
char_skill_dict['will']= char_skill_dict.pop('will') 
char_skill_dict['spell_dc']= char_skill_dict.pop('spell dc')

#view dictionary 
char_skill_dict

{'character build': 'halfling rogue dexterity',
 'level': 2,
 'strength': 0,
 'dexterity': 3,
 'constitution': 1,
 'intelligence': 1,
 'wisdom': 2,
 'charisma': 1,
 'spell attack': 0,
 'hp': 24,
 'ac': 18,
 'speed': 25,
 'percep': 8,
 'fort': 5,
 'ref': 9,
 'will': 8,
 'spell_dc': 10}

In [3]:
def open_clean_actionlist(filepath):
    actionlist = pd.read_csv(filepath)
    #clean up nan calues 
    actionlist= actionlist.replace('x', 0).fillna(0)

    #add a column that calculates changes in location 
    loc_change_list =[]
    for action in range(len(actionlist)):
        action_choice = actionlist.iloc[action]
        if action_choice['action'] == 'move towards target':
            loc_change_list.append(f'+{char_skill_dict['speed']}')
        elif action_choice['action'] == 'move away from target':
            loc_change_list.append(f'-{char_skill_dict['speed']}')
        elif action_choice['action'] == 'flank target':
            loc_change_list.append(f'+{char_skill_dict['speed']}')
        else:
            loc_change_list.append(0)
    
    actionlist['location_change'] = loc_change_list

    #Change data types if needed 
    actionlist['phy_attack_require']= actionlist['phy_attack_require'].astype(int)

    return actionlist

char_actlist = open_clean_actionlist('../build data/rogue_actionlist.csv')
char_actlist.sample(3)

,Cost,target_cond_requirement,phy_attack_require,action,action type,trait,Description,HP bonus,save effect,AC effect,...,range,number die,damage die,additional damage,damage type,duration (round),cool down (round),crit success,crit failure,location_change
16,1,0,0,battle medicine,buff,medicine,heal ally wounds while in battle,0,0,0,...,5,0,0,0,0,0,immune 1 day,0.0,0.0,0
14,0,melee attack,0,nimble dodge,buff,0,"trigger- creature attack, dodge = +2 AC",0,0,2,...,5,0,0,0,0,0,0,0.0,0.0,0
13,1,off-guard,0,dagger sneak attack,buff,dexterity,target must be off-guard,0,0,0,...,5,1,10,3,P,0,0,0.0,0.0,0


### Import data for enemy

In [4]:
monster_build = pd.read_csv('../build data/monster_data_short.csv')
## Clean up the main dataset 
monster_build['name']= monster_build['name'].str.lower()

### Pull out only the stat relevant columns
stat_col = ['name', 'Creature Level', 'Perception',
       'HP', 'AC', 'Fort', 'Ref', 'Will', 'Fort DC', 'Ref DC', 'Will DC',
       'speed_Land','Attack Op']

monster_build= monster_build[stat_col]

#rename columns if needed 
monster_build = monster_build.rename(columns={'speed_Land':'speed'})

#fil in nan with zeros 
monster_build = monster_build.fillna(0)

#make all column names lowercase 
monster_build.columns = monster_build.columns.str.lower()

#view dataframe
monster_build.head(3)

,name,creature level,perception,hp,ac,fort,ref,will,fort dc,ref dc,will dc,speed,attack op
0,abandoned zealot,6,14,75,22,10,14,16,-2.0,0.0,0.0,0,False
1,adlet,10,18,180,30,20,22,16,0.0,0.0,0.0,40,False
2,arbiter,1,7,22,16,5,7,7,-1.0,0.0,0.0,20,False


In [5]:
monster_choice= 'kobold scout'

In [6]:
#Check all monster options to make sure monster of interest is in the list 
monster_list = monster_build['name'].astype('str').unique().tolist()
#make it all lower case 
monster_list = [x.lower() for x in monster_list]
monster_check = monster_choice
monster_name = [x for x in monster_list if re.search(monster_check, x)]
monster_name

#pull out the row for that monster stat 
choice_monster_build = monster_build[monster_build['name'].str.lower() == monster_name[0]]
#make all build options to a dictionary for easier analysis
mon_stat_dict= choice_monster_build.iloc[0].to_dict()

mon_stat_dict

{'name': 'kobold scout',
 'creature level': 1,
 'perception': 8,
 'hp': 16,
 'ac': 18,
 'fort': 5,
 'ref': 9,
 'will': 6,
 'fort dc': -3.0,
 'ref dc': 0.0,
 'will dc': 0.0,
 'speed': '25',
 'attack op': False}

In [7]:
### Pull out all action list items for the chosen monster
mon_actionlist = pd.read_csv('../build data/monster_all_actionlist.csv')

## Filter full monster action list for the monster of interest
enemy_actionlist = mon_actionlist[mon_actionlist['monster_name'] == monster_choice]

#drop old index columns
enemy_actionlist= enemy_actionlist.drop(columns=['Unnamed: 0'])

### Add moving toward and away as movement steps to the monster actionlist 
### Add moving toward and away as movement steps to the monster actionlist 
move_toward_row = [mon_stat_dict['name'], mon_stat_dict['creature level'], 'movement',
       'move towards target', 0, 0, 1,
       'movement', 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]

move_away_row = [mon_stat_dict['name'], mon_stat_dict['creature level'], 'movement',
       'move away from target', 0, 0, 1,
       'movement', 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]

enemy_actionlist.loc[len(enemy_actionlist)] = move_toward_row
enemy_actionlist.loc[len(enemy_actionlist)] = move_away_row
enemy_actionlist = enemy_actionlist.drop_duplicates()
enemy_actionlist= enemy_actionlist.reset_index(drop= True)

#change column names to match the format from the player to make it more standardized
enemy_actionlist= enemy_actionlist.rename(columns={'attack_type':'action type', 'action_name':'action'})

#add a column that calculates changes in location 
loc_change_list =[]
actionlist=enemy_actionlist
for action in range(len(actionlist)):
    action_choice = actionlist.iloc[action]
    if action_choice['action'] == 'move towards target':
        loc_change_list.append(f'-{int(mon_stat_dict['speed'])}')
    elif action_choice['action'] == 'move away from target':
        loc_change_list.append(f'+{int(mon_stat_dict['speed'])}')
    elif action_choice['action'] == 'flank target':
        loc_change_list.append(f'-{int(mon_stat_dict['speed'])}')
    else:
        loc_change_list.append(0)

actionlist['location_change'] = loc_change_list

#view actionlist
enemy_actionlist

,monster_name,monster_level,action_term,action,damage_type,attack_range,action_cost,action type,damage_dice_calculations,attack_roll_mod,spell_DC,spell_slot_level,casting time,target,area,duration,saving throw,spell attack,conditions,location_change
0,kobold scout,1,Attack 1,shortsword,['piercing'],5,1,melee,"[['1', '6', 0]]",9.0,0,0,0,0,0,0,0,0,0,0
1,kobold scout,1,Attack 2,crossbow,['piercing'],120,2,range,"[['1', '8', 0]]",9.0,0,0,0,0,0,0,0,0,0,0
2,kobold scout,1,movement,move towards target,0,0,1,movement,0,0,0,0,0,1,0,0,0,0,0,-25
3,kobold scout,1,movement,move away from target,0,0,1,movement,0,0,0,0,0,1,0,0,0,0,0,+25


----

### Define starting state for the encounter by creating base agent stats as a default
* player and enemy starting stats
* Future edit: include spell slot counter information

In [8]:
### Add additional information to the stats dictionary to better track encounter states
char_skill_dict['agent_name']= char_skill_dict['character build'] 
char_skill_dict['location'] = char_skill_dict['speed']
char_skill_dict['condition']= 'none'

##define it as a new variable
player_base_state= char_skill_dict
player_base_state

{'character build': 'halfling rogue dexterity',
 'level': 2,
 'strength': 0,
 'dexterity': 3,
 'constitution': 1,
 'intelligence': 1,
 'wisdom': 2,
 'charisma': 1,
 'spell attack': 0,
 'hp': 24,
 'ac': 18,
 'speed': 25,
 'percep': 8,
 'fort': 5,
 'ref': 9,
 'will': 8,
 'spell_dc': 10,
 'agent_name': 'halfling rogue dexterity',
 'location': 25,
 'condition': 'none'}

In [9]:
mon_cond_effects='none'
enemy_base_state= {'agent_name': f'{mon_stat_dict['name']}_lvl {mon_stat_dict['creature level']}',
    'location': mon_stat_dict['speed'], 
'hp': mon_stat_dict['hp'],
'ac': mon_stat_dict['ac'],
'speed': mon_stat_dict['speed'],
'percep': mon_stat_dict['perception'],
'fort': mon_stat_dict['fort'],
'ref': mon_stat_dict['ref'],
'will': mon_stat_dict['will'], 
'spell_dc': 0,
'condition': mon_cond_effects}

enemy_base_state

{'agent_name': 'kobold scout_lvl 1',
 'location': '25',
 'hp': 16,
 'ac': 18,
 'speed': '25',
 'percep': 8,
 'fort': 5,
 'ref': 9,
 'will': 6,
 'spell_dc': 0,
 'condition': 'none'}

### Roll for Initiative!

In [10]:
# #character initative= dice roll + perception 
# char_intve = random.randint(1, 20) + player_state['percep'] 
# print("character initative: ", char_intve)
# mon_intve = random.randint(1, 20) + enemy_state['percep'] 
# print("monster initative: ", mon_intve)

----

### Encounter strategy one: Random action selection
* Add feature that updates player/enemy state stats based on conditions. And then conditions are reset between each round
* Filter out action options once they are used in previous round -> so you wont get 3 non-attack actions the same back to back
* Add in code to use buff and spell actions, as well as spell slot counting
* Update monster damage calculations to include multiple attack die (not just the first one)

In [11]:
def update_action_by_location(distance_calculation, action_list):
    agent_distance= distance_calculation
    actlist_filter = action_list
    #If the player and enemy are within 5 feet of eachother -> can use melee actions and movement actions
    if agent_distance<=5:
       #actlist_filter = actlist_filter[actlist_filter['action type'] != 'range'] 
        ### start with only physical attacks, add in buff and spells later
        actlist_filter = actlist_filter[(actlist_filter['action type'] =='melee')|((actlist_filter['action type'] =='movement'))]
    #Else the player can only use range and movement 
    else:
        #actlist_filter = actlist_filter[actlist_filter['action type'] != 'melee']
        ### start with only physical attacks, add in buff and spells later
        actlist_filter = actlist_filter[(actlist_filter['action type'] =='range')|((actlist_filter['action type'] =='movement'))]
    return actlist_filter

######################################################################################
### Simulate one round encounter using random selection 
def one_round_player(action_list, player_dict, enemy_dict, round_number):
    #define necessary variables
    player_state= player_dict 
    enemy_state = enemy_dict
    char_actlist= action_list
    current_turn_player= player_state['agent_name']
    #Create a list to save the results 
    enc_result=[]
    
    #define starting state values 
    ### Check on location -> filter based on location options
    agent_distance = abs(int(enemy_state['location']) + int(player_state['location']))
    target_ac = enemy_state['ac']
    target_health = enemy_state['hp']
    
    #Start the counter for the number of actions available
    action_num= 3
    MAP_counter= 1
    #dictionary to adjust attack penalties
    MAP_dict={1:0, 2:-5, 3:-10}
    
    ### Simulate one round of combat ----------------------------------------------
    while action_num >0:
    #### Selection action from avaiable actions ###
        #Filter based on player distance 
        actlist_filter= update_action_by_location(agent_distance, char_actlist)
        #Filter based on action economy 
        actlist_filter= actlist_filter[(actlist_filter['phy_attack_require']==MAP_counter) | (actlist_filter['phy_attack_require']==0)]
        #Filter based on whether the enemy is within the attack range
        actlist_filter = actlist_filter[(actlist_filter['range '].astype(int) >= agent_distance) | (actlist_filter['range '].astype(int)==0)]
        #filter based on condition or other requirements ###################
        
        ### Random Choice ###
        #Pick an avaiable action by random 
        attack_choice = actlist_filter.sample(n=1).iloc[0]
    
        ### Simulate the results of that action
        #calculate the chance of hitting 
        roll_calc = int(attack_choice['attack roll']) + random.randint(1, 20)
        #roll_calc = 30
        crit_level = max(int(attack_choice['attack roll']) + 20, int(target_ac) + 10)
        crit_fail_level = int(attack_choice['attack roll']) + 1
        
        #results for a normal hit
        if roll_calc >= target_ac and roll_calc <= crit_level:
            #print("sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + int(attack_choice['additional damage'])
            attack_sucess = 1
        #results for a critical hit 
        elif roll_calc >=target_ac and roll_calc >= crit_level:
           # print("crit sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + (num_dmg_die * random.randint(1, die_max))+ int(attack_choice['additional damage'])
            attack_sucess = 2
        #results for a critical fail 
        elif roll_calc < target_ac and roll_calc <= crit_fail_level:
            #print("crit fail") #calculate total damage 
            total_damage = 0
            attack_sucess = -1
        #results for a normal fail
        elif roll_calc < target_ac:
            #print('fail')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        #store read errors as failures
        else:
            #print('other')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        
        ## Now calculate the state changes from the roll and save results 
        
        ######calculate the 'reward' or change in state based on these results #####
        #Current health or AC changes based on the results 
        target_health = target_health - total_damage
        
        #calculate action reward
        attack_reward= attack_sucess*total_damage
        if target_health <=0:
            attack_reward= attack_reward + 100
    
        ### Update encounter states:
        # ## Update location 
        # if attack_choice['location_change'] != 0:
        #     total_location_change = int(attack_choice['location_change'])
        #     agent_distance= agent_distance+ total_location_change
        
        ## Add in conditions and update other components of player-enemy states
        enemy_state['condition']= (attack_choice['target effect'])
        #update location information
        player_state['location']= int(player_state['location']) + int(attack_choice['location_change'])
        agent_distance = abs(int(enemy_state['location']) + int(player_state['location']))
       
         #Save the results in the results_list 
        enc_result.append([round_number, current_turn_player, action_num, attack_choice['action'], attack_choice['action type'], 
                           agent_distance, attack_sucess, total_damage, target_health, attack_reward])
        
        #Calculate the action cost 
        action_num = action_num - attack_choice['Cost']
    
        #calculate the MAP cost if it was a melee or range attack
        if attack_choice['action type'] == 'melee' or attack_choice['action type'] == 'range':
            MAP_counter+=1

    ### After the round is complete
    ## Update player/enemy states based on action results
    enemy_state['hp'] = target_health
    ### Change state based on conditions --> Will need to code a conditions dictionary here
    
    ##After one round of player combat ==> Return the results and each of the player and enemy state
    
    return enc_result, player_state, enemy_state

def one_round_enemy(action_list, player_dict, enemy_dict, round_number):
    #define necessary variables
    enemy_state = enemy_dict
    player_state= player_dict
    char_actlist= action_list
    current_turn_player= enemy_state['agent_name']
    
    #Create a list to save the results 
    enc_result=[]
    
    #define starting state values 
    ### Check on location -> filter based on location options
    agent_distance = abs(int(enemy_state['location']) + int(player_state['location']))
    target_ac =player_state['ac']
    target_health = player_state['hp']
    
    #Start the counter for the number of actions available
    action_num= 3
    MAP_counter= 1
    MAP_dict={1:0, 2:-5, 3:-10}
    
    ### Simulate one round of combat ----------------------------------------------
    while action_num >0:
    #### Selection action from avaiable actions ###
        #Filter based on player distance 
        actlist_filter= update_action_by_location(agent_distance, char_actlist)
        #Adjusted attack roll bonus based on the multiple-attack-penalty 
        #actlist_filter= actlist_filter[(actlist_filter['phy_attack_require']==MAP_counter) | (actlist_filter['phy_attack_require']==0)]
        actlist_filter['attack_roll_mod']= actlist_filter['attack_roll_mod'].astype(float)+MAP_dict[int(MAP_counter)]
        #Filter based on whether the enemy is within the attack range
        actlist_filter = actlist_filter[(actlist_filter['attack_range'].astype(int) >= agent_distance) | (actlist_filter['attack_range'].astype(int)==0)]
        #filter based on condition or other requirements ###################
        
        ### Random Choice ###
        #Pick an avaiable action by random 
        attack_choice = actlist_filter.sample(n=1).iloc[0]
    
        ### Simulate the results of that action
        #calculate the chance of hitting 
        roll_calc = int(attack_choice['attack_roll_mod']) + random.randint(1, 20)
        #roll_calc = 30
        crit_level = max(int(attack_choice['attack_roll_mod']) + 20, int(target_ac) + 10)
        crit_fail_level = int(attack_choice['attack_roll_mod']) + 1
    
        #convert the format of the damage dice calculations to a formula
        dice_list = attack_choice['damage_dice_calculations']
    
        #check if it is a damage type action 
        if dice_list != 0:
            dice_calc_num = re.findall(r'\d+',dice_list)
            dice_calc_num
            if len(dice_calc_num) >3:
                print("more damage from dice to calculate")
        
            
            #results for a normal hit
            if roll_calc >= target_ac and roll_calc <= crit_level:
                die_max = int(dice_calc_num[1])
                if die_max ==0: 
                    die_max = 4
                num_dmg_die= int(float(dice_calc_num[0]))
                add_attack= int(dice_calc_num[2])
                total_damage = (num_dmg_die * random.randint(1, die_max)) + add_attack
                attack_sucess = 1
            #results for a critical hit 
            elif roll_calc >=target_ac and roll_calc >= crit_level:
               # print("crit sucess") #calculate total damage 
                die_max = int(dice_calc_num[1])
                if die_max ==0: 
                    die_max = 4
                num_dmg_die= int(float(dice_calc_num[0]))
                add_attack= int(dice_calc_num[2])
                total_damage = (num_dmg_die * random.randint(1, die_max)) + (num_dmg_die * random.randint(1, die_max))+ add_attack
                attack_sucess = 2
            #results for a critical fail 
            elif roll_calc < target_ac and roll_calc <= crit_fail_level:
                #print("crit fail") #calculate total damage 
                total_damage = 0
                attack_sucess = -1
            #results for a normal fail
            elif roll_calc < target_ac:
                #print('fail')
                total_damage = 0
                attack_sucess = 0
                #print('total damage =', total_damage)
            #store read errors as failures
            else:
                #print('other')
                total_damage = 0
                attack_sucess = 0
        else:
            total_damage = 0
            attack_sucess = 0
        
        ## Now calculate the state changes from the roll and save results 
        
        ######calculate the 'reward' or change in state based on these results #####
        #Current health or AC changes based on the results 
        target_health = target_health - total_damage
        
        #calculate action reward
        attack_reward= attack_sucess*total_damage
        if target_health <=0:
            attack_reward= attack_reward + 100
    
        ### Update encounter states:
        # ## Update location 
        # if attack_choice['location_change'] != 0:
        #     total_location_change = int(attack_choice['location_change'])
        #     agent_distance= agent_distance+ total_location_change
        
        ## Add in conditions
        player_state['condition']= (attack_choice['conditions'])
        #update location info
        enemy_state['location']= int(enemy_state['location']) + int(float(attack_choice['location_change']))
        agent_distance = abs(int(enemy_state['location']) + int(player_state['location']))
        #Save the results in the results_list 
        enc_result.append([round_number, current_turn_player, action_num, attack_choice['action'], attack_choice['action type'], 
                           agent_distance, attack_sucess, total_damage, target_health, attack_reward])
        
        #Calculate the action cost 
        action_num = action_num - int(attack_choice['action_cost'])
    
        #calculate the MAP cost if it was a melee or range attack
        if attack_choice['action type'] == 'melee' or attack_choice['action type'] == 'range':
            MAP_counter+=1
    
    ### After the round is complete
    ## Update player/enemy states based on action results
    player_state['hp'] = target_health
    ### Change state based on conditions --> Will need to code a conditions dictionary here
    
    ##After one round of player combat ==> Return the results and each of the player and enemy state
    return enc_result, player_state, enemy_state

## Run simulation for a single encounter 

In [12]:
# ####
# #Starting encounter reset
# ####
# #Reset player starting state 
# player_cond_effects='none'
# player_state = {'agent_name': char_skill_dict['character build'],
#     'location': char_skill_dict['speed'], 
# 'hp': char_skill_dict['HP'],
# 'ac': char_skill_dict['AC'],
# 'speed': char_skill_dict['speed'],
# 'percep': char_skill_dict['perception'],
# 'fort': char_skill_dict['fortitude'],
# 'ref': char_skill_dict['reflex'],
# 'will': char_skill_dict['will'], 
# 'spell_dc': char_skill_dict['spell dc'],
# 'condition': player_cond_effects}

# #reset enemy starting state
# mon_cond_effects='none'
# enemy_state= {'agent_name': f'{mon_stat_dict['name']}_lvl {mon_stat_dict['creature level']}',
#     'location': mon_stat_dict['speed'], 
# 'hp': mon_stat_dict['hp'],
# 'ac': mon_stat_dict['ac'],
# 'speed': mon_stat_dict['speed'],
# 'percep': mon_stat_dict['perception'],
# 'fort': mon_stat_dict['fort'],
# 'ref': mon_stat_dict['ref'],
# 'will': mon_stat_dict['will'], 
# 'spell_dc': 0,
# 'condition': mon_cond_effects}

# ## Define player starting position ### Define starting locations as one movement away from eachother 
# player_state.update({'location': -player_state['speed']})
# enemy_state.update({'location': enemy_state['speed']})

# ### Roll for intiative ### 
# #character initative= dice roll + perception 
# char_intve = random.randint(1, 20) + player_state['percep'] 
# #print("character initative: ", char_intve)
# mon_intve = random.randint(1, 20) + enemy_state['percep'] 
# #print("monster initative: ", mon_intve)

# ####
# # Start encounter: random selection 
# ####

# #Save results to a main dataframe 
# final_col_names = columns= ['round_number', 'agent', 'round_action_num', 'action_name', 'action_type', 
#                            'total_agent_distance', 'attack_sucess', 'total_damage', 'target_health', 'attack_reward']
# full_enc_results= pd.DataFrame(columns= final_col_names)

# ### Simulate the results from one full encounter
# if char_intve >= mon_intve:
#     round_number = 1
#     while enemy_state['hp'] >0 or player_state['hp'] >0:
#         round_actions, player_state, enemy_state = one_round_player(char_actlist, player_state, enemy_state, round_number)
#         full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
#         #print(f'{round_number}: monster HP {enemy_state['hp']}')
    
#         #End the loop once the monster dies
#         if enemy_state['hp'] < 0: 
#             print(f"Player defeats monster in {round_number} rounds")
#             break
    
#         round_actions, player_state, enemy_state = one_round_enemy(enemy_actionlist, player_state, enemy_state, round_number)
#         full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
#         #print(f'{round_number}: character HP {player_state['hp']}')
    
#         if player_state['hp'] < 0: 
#             print(f"monster defeats player in {round_number} rounds")
#             break
#         round_number +=1

# elif  mon_intve > char_intve:
#     round_number = 1
#     while enemy_state['hp'] >0 or player_state['hp'] >0:
#         round_actions, player_state, enemy_state = one_round_enemy(enemy_actionlist, player_state, enemy_state, round_number)
#         full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
#         #print(f'{round_number}: character HP {player_state['hp']}')
    
#         if player_state['hp'] < 0: 
#             print(f"monster defeats player in {round_number} rounds")
#             break
            
#         round_actions, player_state, enemy_state = one_round_player(char_actlist, player_state, enemy_state, round_number)
#         full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
#         #print(f'{round_number}: monster HP {enemy_state['hp']}')
    
#         #End the loop once the monster dies
#         if enemy_state['hp'] < 0: 
#             print(f"Player defeats monster in {round_number} rounds")
#             break
#         round_number +=1

# full_enc_results

----

### Add in a reference dictionary for status conditions and bonus calculations

In [13]:
### Create the conditions chart 
cond_status_dict = {'enfeebled': {'attack_roll':-1, 'athletics':-1, 'strength': -1, 'damage_rolls':-1}, 
 'fleeing': {'num_actions': +3, 'location_change': -90}, 
 'stunned': {'num_actions': +1}, 
 'deafened': {'perception': -2}, 
 'invisible': {'total_damage': 0}, 
 'encumbered': {'ac': -1, 'reflex': -1, 'attack_roll': -1, 'speed': -10}, 
 'clumsy': {'ac': -1, 'reflex': -1, 'attack_roll': -1}, 
 'stupefied': {'will': -1, 'spell_attack_mod': -1, 'spell_dc': -1}, 
 'confused': {'ac': -2, 'conditions': 'off-gaurd'}, 
 'blinded': {'perception':-4, 'speed': 15}, 
 'prone': {'ac': -2, 'attack_roll_mod': -2, 'num_actions': +1, 'conditions': 'off-guard'}, 
 'slowed': {'num_actions': +1}, 
 'dazzled': {'attack_roll_mod': -1, 'spell_attack': -1}, 
 'grabbed': {'ac': -2, 'condition': 'off_gaurd'}, 
 'immobilized': {'speed': 0},
 'restrained': {'ac': -2, 'num_actions': +1, 'conditions': 'off-guard'}, 
 'persistent damage': {'damage_rolls': 'persistant_dice_roll', 'cool_down': 'whole_encounter'},
 'paralyzed': {'ac':-2, 'conditions': 'off_gaurd', 'speed':0, 'num_actions': +3}, 
 'quickened': {'num_actions':-1},
 'fascinated':{'perception': -2},
 'drained': {'con': -1}, 
 'fatigued': {'ac':-1, 'fort': -1, 'ref': -1, 'will': -1, 'cool_down': 'whole_encounter'},
 'controlled': {'num_actions': +3}, 
 'petrified': {'num_actions': +3, 'speed': 0}, 
 'sickened': {'fort':-1, 'ref': -1, 'will': -1}, 
 'rage': {'hp':'hp+con', 'damage_roll': +2, 'cool_down': 'whole_encounter' }
}
#view final dictionary
#cond_status_dict

---

### Simulate 20 different encounters for a player-monster pair and calculate encounter rewards

In [14]:
#start off with an encounter ticker
encounter_num= 1
max_round_per_encounter = 100
#Final encounter summary results 
enc_result_log=[]
#save final dataframe of all round decisions
final_col_names = columns= ['round_number', 'agent', 'round_action_num', 'action_name', 'action_type', 
                           'total_agent_distance', 'attack_sucess', 'total_damage', 'target_health', 'attack_reward']
enc_action_log=pd.DataFrame(columns=final_col_names)

#while encounter_num <=20:
for _ in range(20):
    ####
    #Starting encounter reset
    ####
    #Reset player starting state 
    player_cond_effects='none'
    player_state = {'agent_name': char_skill_dict['character build'], 
                    'location': char_skill_dict['speed'], 
    'hp': char_skill_dict['hp'],
    'ac': char_skill_dict['ac'],
    'speed': char_skill_dict['speed'],
    'percep': char_skill_dict['percep'],
    'fort': char_skill_dict['fort'],
    'ref': char_skill_dict['ref'],
    'will': char_skill_dict['will'], 
    'spell_dc': char_skill_dict['spell_dc'],
    'condition': player_cond_effects}
    
    #reset enemy starting state
    mon_cond_effects='none'
    enemy_state= {'agent_name': f'{mon_stat_dict['name']}_lvl {mon_stat_dict['creature level']}',
        'location': mon_stat_dict['speed'], 
    'hp': mon_stat_dict['hp'],
    'ac': mon_stat_dict['ac'],
    'speed': mon_stat_dict['speed'],
    'percep': mon_stat_dict['perception'],
    'fort': mon_stat_dict['fort'],
    'ref': mon_stat_dict['ref'],
    'will': mon_stat_dict['will'], 
    'spell_dc': 0,
    'condition': mon_cond_effects}
    
    ## Define player starting position ### Define starting locations as one movement away from eachother 
    player_state.update({'location': -player_state['speed']})
    enemy_state.update({'location': enemy_state['speed']})
    
    ### Roll for intiative ### 
    #character initative= dice roll + perception 
    char_intve = random.randint(1, 20) + player_state['percep'] 
    #print("character initative: ", char_intve)
    mon_intve = random.randint(1, 20) + enemy_state['percep'] 
    #print("monster initative: ", mon_intve)
    
    ####
    # Start encounter: random selection 
    ####
    
    #Save results to a main dataframe 
    full_enc_results= pd.DataFrame(columns= final_col_names)
    
    ### Simulate the results from one full encounter
    if char_intve >= mon_intve:
        round_number = 1
        while enemy_state['hp'] >=0 or player_state['hp'] >=0 or round_number <=max_round_per_encounter:
            round_actions, player_state, enemy_state = one_round_player(char_actlist, player_state, enemy_state, round_number)
            full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
            #print(f'{round_number}: monster HP {enemy_state['hp']}')
        
            #define an end statement
            if enemy_state['hp'] < 0 or round_number >=max_round_per_encounter: 
                enemy_state['hp'] == 0
                #print(f"{player_state['agent_name']} defeats {enemy_state['agent_name']} in {round_number} rounds")
                #print(f"encounter {encounter_num} simulation complete in {round_number} rounds")
                break
        
            round_actions, player_state, enemy_state = one_round_enemy(enemy_actionlist, player_state, enemy_state, round_number)
            full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
            #print(f'{round_number}: character HP {player_state['hp']}')

            #define an end statement
            if player_state['hp'] < 0 or round_number >=max_round_per_encounter: 
                player_state['hp'] == 0
                #print(f"{enemy_state['agent_name']} defeats {player_state['agent_name']} in {round_number} rounds")
                #print(f"encounter {encounter_num} simulation complete in {round_number} rounds")
                break
            
            round_number +=1
    
    elif mon_intve > char_intve:
        round_number = 1
        while enemy_state['hp'] >=0 or player_state['hp'] >=0 or round_number <=max_round_per_encounter:
            round_actions, player_state, enemy_state = one_round_enemy(enemy_actionlist, player_state, enemy_state, round_number)
            full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
        
            #define an end statement
            if player_state['hp'] <= 0 or round_number >=100: 
                player_state['hp']==0
                #print(f"{enemy_state['agent_name']} defeats {player_state['agent_name']} in {round_number} rounds")
                #print(f"encounter {encounter_num} simulation complete in {round_number} rounds")
                break
                
            round_actions, player_state, enemy_state = one_round_player(char_actlist, player_state, enemy_state, round_number)
            full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions, columns= final_col_names)])
            #print(f'{round_number}: monster HP {enemy_state['hp']}')
        
            #define an end statement
            if enemy_state['hp'] <= 0 or round_number >=100: 
                enemy_state['hp'] == 0
                #print(f"{player_state['agent_name']} defeats {enemy_state['agent_name']} in {round_number} rounds")
                #print(f"encounter {encounter_num} simulation complete in {round_number} rounds")
                break
            round_number +=1
    
    ### Save encounter simulation results

    #Add full encounter actions to the main encounter log
    enc_action_log= pd.concat([enc_action_log, full_enc_results])

    ##calculate an encounter reward: (+1 if player wins, -1 if monster wins) / number of rounds for combat
    ## Therefore shorter rounds of combat from the players will get a BIGGER final reward
    total_round_counter = int(full_enc_results['round_number'].max())
    encounter_winner = full_enc_results.iloc[-1]['agent']
    
    if encounter_winner == f'{player_state['agent_name']}':
        encounter_reward = round(1 / total_round_counter, 2)
    elif encounter_winner == f'{enemy_state['agent_name']}':
        ### Conceptually if the player lasted a long time (higher round counter) that would be GOOD because it was not immeditely defeated -> 
        ### so would need to adjust the reward calculation to reflect that, which means this calculation is not ideal
        encounter_reward = round(-1 / total_round_counter, 2)
    else:
        #creating a non-reward because this would only happen if the encounter simulator or encoutner reward filters had an error
        encounter_reward= 0 
    
    #save final encounter calculations to summary log
    enc_result_log.append([encounter_num, f'{player_state['agent_name']}', f'{enemy_state['agent_name']}', encounter_winner, total_round_counter, (encounter_reward)*100])

    #increase encounter ticker and repeat simulation 
    #print(f"encounter simulation {encounter_num} complete")
    encounter_num+=1
        
#visualize the final results of the encounter simulation 
pd.DataFrame(enc_result_log, columns=['encounter_num', 'player_agent', 'enemy_agent', 'encounter_winner', 'total_round_counter', 'encounter_reward'])

,encounter_num,player_agent,enemy_agent,encounter_winner,total_round_counter,encounter_reward
0,1,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,37,-3.0
1,2,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,11,-9.0
2,3,halfling rogue dexterity,kobold scout_lvl 1,halfling rogue dexterity,34,3.0
3,4,halfling rogue dexterity,kobold scout_lvl 1,halfling rogue dexterity,30,3.0
4,5,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,42,-2.0
5,6,halfling rogue dexterity,kobold scout_lvl 1,halfling rogue dexterity,10,10.0
6,7,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,100,-1.0
7,8,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,10,-10.0
8,9,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,11,-9.0
9,10,halfling rogue dexterity,kobold scout_lvl 1,kobold scout_lvl 1,48,-2.0


----